# 01 · Understand, validate and prepare the data

Actual Superstore sample data, not simulated outputs. Run cells from top to bottom. Dollar amounts are USD.


In [1]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/project.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the extracted project folder.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.project import load, summarize, save_table, chart
pd.set_option('display.max_columns',30)
pd.set_option('display.width',160)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(10,5),'axes.spines.top':False,'axes.spines.right':False})

## Business question and grain
Where are sales profitable, where are losses concentrated, and what should management investigate?
One row is an **order line**, not an order. Multiple rows may share Order ID. Never deduplicate Order ID or count rows as orders. The source is a historical sample dataset, not current company performance.

In [2]:
raw = pd.read_csv(ROOT/'data/raw/Superstore.csv',encoding='latin1',dtype={'Postal Code':str})
print('Shape:',raw.shape)
print(raw.head(3).to_string(index=False))
print(raw.dtypes.to_string())

Shape: (9994, 21)
 Row ID       Order ID Order Date  Ship Date    Ship Mode Customer ID   Customer Name   Segment       Country        City      State Postal Code Region      Product ID        Category Sub-Category                                                Product Name  Sales  Quantity  Discount   Profit
      1 CA-2016-152156  11/8/2016 11/11/2016 Second Class    CG-12520     Claire Gute  Consumer United States   Henderson   Kentucky       42420  South FUR-BO-10001798       Furniture    Bookcases                           Bush Somerset Collection Bookcase 261.96         2       0.0  41.9136
      2 CA-2016-152156  11/8/2016 11/11/2016 Second Class    CG-12520     Claire Gute  Consumer United States   Henderson   Kentucky       42420  South FUR-CH-10000454       Furniture       Chairs Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back 731.94         3       0.0 219.5820
      3 CA-2016-138688  6/12/2016  6/16/2016 Second Class    DV-13045 Darrin Van Huff Corporate United 

## Audit before cleaning
Keep the original file unchanged. Remove exact duplicate rows only; conflicting Row IDs are an error requiring review. Negative profit is meaningful and must remain. Dates use month/day/year. Postal codes are text, preserving leading zeros.

In [3]:
audit = pd.DataFrame({'dtype':raw.dtypes.astype(str),'missing':raw.isna().sum(),'unique_values':raw.nunique()})
print(audit)
print('Exact duplicate rows:',raw.duplicated().sum())
save_table(audit,'data_quality')
df = raw.drop_duplicates().copy()
for col in df.select_dtypes(include=['object','string']).columns:
    df[col] = df[col].str.strip()
for col in ['Order Date','Ship Date']:
    df[col] = pd.to_datetime(df[col],format='%m/%d/%Y',errors='raise')
for col in ['Sales','Profit','Quantity','Discount']:
    df[col] = pd.to_numeric(df[col],errors='raise')
required=['Row ID','Order ID','Customer ID','Product ID','Order Date','Ship Date','Sales','Profit','Quantity','Discount','Category','Sub-Category','Region','State']
assert df[required].notna().all().all(), 'Missing required values; investigate before continuing.'
assert df['Row ID'].is_unique, 'Conflicting line identifiers.'
assert np.isfinite(df[['Sales','Profit','Quantity','Discount']].to_numpy()).all()
assert df['Sales'].ge(0).all()
assert (df['Quantity'].gt(0) & df['Quantity'].mod(1).eq(0)).all()
assert df['Discount'].between(0,1,inclusive='left').all()
assert (df['Ship Date']>=df['Order Date']).all()
assert df.groupby('Order ID')[['Customer ID','Order Date']].nunique().le(1).all().all()
df['Postal Code']=df['Postal Code'].str.zfill(5)
df['Year']=df['Order Date'].dt.year
df['Month Start']=df['Order Date'].dt.to_period('M').dt.to_timestamp()
df['Ship Days']=(df['Ship Date']-df['Order Date']).dt.days
df['Loss Flag']=df['Profit'].lt(0).astype(int)
df['Discount Band']=pd.cut(df['Discount'],[-0.001,0,0.2,0.4,1],labels=['0%','>0–20%','>20–40%','>40%'])
df.to_csv(ROOT/'data/processed/sales_clean.csv',index=False,date_format='%Y-%m-%d')
print('Validated and saved',len(df),'order lines.')
print('Period:',df['Order Date'].min().date(),'to',df['Order Date'].max().date())

                 dtype  missing  unique_values
Row ID           int64        0           9994
Order ID        object        0           5009
Order Date      object        0           1237
Ship Date       object        0           1334
Ship Mode       object        0              4
Customer ID     object        0            793
Customer Name   object        0            793
Segment         object        0              3
Country         object        0              1
City            object        0            531
State           object        0             49
Postal Code     object        0            631
Region          object        0              4
Product ID      object        0           1862
Category        object        0              3
Sub-Category    object        0             17
Product Name    object        0           1850
Sales          float64        0           5825
Quantity         int64        0             14
Discount       float64        0             12
Profit       

## Limitations
No returns, delivery date, marketing spend, stock levels, or explicit cost breakdown exist. Ship Days measures time to dispatch, not delivery speed. Profit is supplied by the source; we cannot verify its accounting definition. Do not invent return rates or customer acquisition costs.